# PhysAR-IDS -- NB3c (v6): a true white-box attack on the tuned XGBoost

**Why (reviewer issue 1).** The reviewer is right that the paper's "white-box" labels do not fit: NB3 (PGD) attacks a *surrogate MLP* and transfers the result to XGBoost, and NB4 (PPO) sees only the hard label. Neither reads the target model. This notebook adds the missing attack: the attacker is given the **trees themselves** (every split threshold and leaf value of the Optuna-tuned XGBoost, per seed) and searches directly on the target's own score. It is the reference point for what a white-box attacker achieves against the same detector, rows and budgets as PPO (NB4b / NB4e), so the dissertation can name each attack for what it is.

**Attack ("tree-greedy").** XGBoost is piecewise constant in every feature: the score only changes where the value crosses a split threshold of some tree. For one row the attacker therefore tries, for each controllable feature, a move to **each cell between consecutive thresholds** inside the allowed interval (the nearest `K` thresholds in each direction plus the interval ends), reads the model's margin for every candidate, and applies the single best move. It repeats up to `MAX_ITERS` times, one feature per round, until the row is classified benign (`margin(benign) - max margin(other class) > 0`) or no single move helps. Moves go to the *nearest edge* of a cell, so the perturbation is the smallest that crosses the threshold. Nothing is sampled and nothing is learned: given the trees it is deterministic.

**What it is not.** It is a greedy coordinate search, not an exact optimiser (an exact white-box attack on tree ensembles is a MILP: Kantchelian et al., 2016). Its evasion rate is therefore a **lower bound** on what a white-box attacker can reach at that budget. The sanity check is the unlimited-budget setting, which should reach ~1.0.

**Threat model, same as NB4b / NB4c.** Controllable features: the 60 cyber features (`cyber_*`), each moved within an L-infinity ball of `budget` sigma around the row's own value (features are standardised, so 1 = one training standard deviation), and never outside the observed training range [min, max] (a value already outside it may not go further out). Only attack rows are perturbed; benign rows keep their clean predictions. Settings: budgets 0.1 / 0.2 / 0.3 / 0.5 / 1.0 sigma and "inf" (training range only). One extra setting, `all93_0.1`, lets the attacker move **all 93 features** by 0.1 sigma independently, exactly the feature freedom of NB3's PGD (which also ignores physical consistency), to compare white-box with the transfer attack on equal footing.

**Reported per setting and seed:** per-class evasion (predicted benign after the attack, by the CPU oracle as in NB4), full-test weighted/macro F1 and per-class recall (as NB4e), benign precision, features changed per row, size of the change, and whether the search stopped because no move helped or because of the iteration limit. Comparisons: PPO and random search (from NB4e's saved results), PGD (from NB3b's saved confusion matrices), paired over the 5 seeds.

**Runtime.** Every candidate is scored by the real model, so this is heavier than NB4e: expect on the order of an hour or two on the GPU for all 7 settings x 5 seeds; progress prints per round; results are saved per seed and setting and the run resumes.

**v6b change (burst intervals).** Same attack, same numbers (it is deterministic). Two additions: the per-row evasion flag is now saved for every seed and setting, and section 6b computes 95% intervals by resampling attack bursts (the same unit as NB4), pooled over seeds. The cache fingerprint changed to `nb3c-v2`, so every seed is rerun once.

## 0. Setup

In [ ]:
import os, sys, time, json, hashlib, warnings
import numpy as np, pandas as pd, joblib
import xgboost as xgb
from scipy import stats
from sklearn.metrics import f1_score, precision_score, recall_score
import matplotlib; import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False
K_CAND = 12            # nearest split thresholds tried in each direction per feature (plus the two interval ends)
MAX_ITERS = 25         # greedy rounds (one feature moved per row per round)
CHUNK = 20000          # rows per search block
BLOCK = 1_000_000      # candidate rows scored per model call
# setting name -> (feature group, cumulative L-inf budget in sigma, tag used by NB4e for the PPO reference)
SETTINGS = {'cyber_0.1': ('cyber', 0.1, '0.1'), 'cyber_0.2': ('cyber', 0.2, '0.2'), 'cyber_0.3': ('cyber', 0.3, '0.3'), 'cyber_0.5': ('cyber', 0.5, '0.5'),
            'cyber_1.0': ('cyber', 1.0, '1.0'), 'cyber_inf': ('cyber', None, 'unconstrained'), 'all93_0.1': ('all', 0.1, None)}
SETTINGS_TO_RUN = list(SETTINGS)

os.environ['PYTHONHASHSEED'] = '42'
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
print(f'xgboost {xgb.__version__} | search backend: {"cpu(hist)" if _cpu else "gpu(cuda,hist)"} | working directory: {os.getcwd()}')

## 1. Load v6 data and the NB2 tuned configuration

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]; ALL_IDX = np.arange(N_FEAT)
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)
atk_te = y_test != 0; X_atk_test, y_atk_test = X_test[atk_te], y_test[atk_te]
test_burst = pd.factorize(pd.MultiIndex.from_arrays([y_test, np.asarray(data['PLCID_test']), np.asarray(data['timestamp_test'])]))[0][atk_te]
FEAT_LO, FEAT_HI = X_train.min(0), X_train.max(0)
print(f'Features {N_FEAT} | cyber (controllable in the main settings) {len(CYBER_IDX)} | evaluation attack rows {X_atk_test.shape} | class counts {np.bincount(y_atk_test)[1:]}')
nb2 = joblib.load(f'optuna_xgboost_results_{PIPELINE_VERSION}.pkl')
nb2_tuned = np.array([nb2['results']['tuned'][s]['test_f1'] for s in SEEDS if s in nb2['results']['tuned']])
print(f'NB2 tuned test F1 over seeds: {nb2_tuned.mean():.4f} +/- {nb2_tuned.std(ddof=1):.4f}')

## 2. The attack
`get_thresholds` reads every split value from the fitted trees. XGBoost sends a row left when `x < threshold`, so crossing *up* means setting the feature to the threshold itself and crossing *down* means setting it to the float just below it (`nextafter`). `one_step` scores every candidate move with the model's own margin and returns, per row, the best one; `whitebox` repeats it.

In [ ]:
def get_thresholds(clf):
    """feature index -> sorted unique float32 split thresholds used anywhere in the ensemble."""
    df = clf.get_booster().trees_to_dataframe(); sp = df[df['Feature'] != 'Leaf']
    return {int(str(f).lstrip('f')): np.unique(g['Split'].to_numpy(dtype=np.float32)) for f, g in sp.groupby('Feature')}

class Margin:
    """Raw per-class scores of a fitted classifier (the white-box access: the attacker evaluates the model's own trees)."""
    def __init__(self, clf): self.b = clf.get_booster()
    def __call__(self, X):
        out = []
        for s in range(0, len(X), BLOCK):
            z = self.b.inplace_predict(np.ascontiguousarray(X[s:s + BLOCK], dtype=np.float32), predict_type='margin', validate_features=False)
            out.append(np.asarray(z))
        return np.concatenate(out) if len(out) > 1 else out[0]
def gap_of(z): return (z[:, 0] - z[:, 1:].max(1)).astype(np.float64)       # > 0  <=>  benign is the arg-max class

def interval(o, j, B):
    """allowed interval of feature j for rows with original values o: L-inf ball (if B) intersected with the training range (never further out than the row already is)."""
    lo_r, hi_r = np.minimum(FEAT_LO[j], o), np.maximum(FEAT_HI[j], o)
    if B is None: return lo_r.astype(np.float32), hi_r.astype(np.float32)
    return np.maximum(o - np.float32(B), lo_r).astype(np.float32), np.minimum(o + np.float32(B), hi_r).astype(np.float32)

def candidates(xc, X0c, j, T, B, K):
    """candidate (row, value) moves for feature j: the K nearest thresholds on each side inside the allowed interval, plus the interval ends."""
    xv = xc[:, j]; a, b = interval(X0c[:, j], j, B); nT = len(T); idx = np.searchsorted(T, xv, side='right'); R, V = [], []
    for d in range(min(K, nT)):
        k = idx + d; v = T[np.minimum(k, nT - 1)]; ok = (k < nT) & (v <= b); R.append(np.flatnonzero(ok)); V.append(v[ok])
        k = idx - 1 - d; v = np.nextafter(T[np.maximum(k, 0)], np.float32(-np.inf)); ok = (k >= 0) & (v >= a); R.append(np.flatnonzero(ok)); V.append(v[ok])
    ok = b > xv; R.append(np.flatnonzero(ok)); V.append(b[ok]); ok = a < xv; R.append(np.flatnonzero(ok)); V.append(a[ok])
    return np.concatenate(R), np.concatenate(V).astype(np.float32)

def one_step(margin, xc, X0c, g, feats, TH, B, K):
    """best single-feature move per row (by the model's own margin gap). Returns best_gap, best_feature (-1 = none helps), best_value, n_scored."""
    n = len(xc); best_g = g.copy(); best_j = np.full(n, -1, np.int64); best_v = np.zeros(n, np.float32); n_scored = 0
    bufR, bufJ, bufV = [], [], []; size = 0
    def flush():
        nonlocal n_scored
        if not bufR: return
        R, J, V = np.concatenate(bufR), np.concatenate(bufJ), np.concatenate(bufV); bufR.clear(); bufJ.clear(); bufV.clear()
        for s in range(0, len(R), BLOCK):
            r, jj, v = R[s:s + BLOCK], J[s:s + BLOCK], V[s:s + BLOCK]
            Xc = xc[r]; Xc[np.arange(len(r)), jj] = v                    # xc[r] is a copy
            gg = gap_of(margin(Xc)); n_scored += len(r)
            o = np.argsort(r, kind='stable'); rs, gs = r[o], gg[o]
            starts = np.flatnonzero(np.r_[True, rs[1:] != rs[:-1]]); segmax = np.maximum.reduceat(gs, starts); cnt = np.diff(np.r_[starts, len(rs)])
            hit = np.flatnonzero(gs == np.repeat(segmax, cnt)); _, first = np.unique(rs[hit], return_index=True); sel = o[hit[first]]
            rows = r[sel]; better = gg[sel] > best_g[rows]; rr = rows[better]
            best_g[rr] = gg[sel][better]; best_j[rr] = jj[sel][better]; best_v[rr] = v[sel][better]
    for j in feats:
        T = TH.get(int(j))
        if T is None: continue
        R, V = candidates(xc, X0c, int(j), T, B, K)
        if len(R) == 0: continue
        bufR.append(R); bufJ.append(np.full(len(R), j, np.int64)); bufV.append(V); size += len(R)
        if size >= BLOCK: flush(); size = 0
    flush(); return best_g, best_j, best_v, n_scored

def whitebox(margin, X0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):
    x = X0.copy(); g = gap_of(margin(x)); init_evaded = g > 0; active = ~init_evaded; stalled = np.zeros(len(x), bool); log = []
    for it in range(max_iters):
        act = np.flatnonzero(active)
        if len(act) == 0: break
        t0 = time.time(); scored = 0
        for s in range(0, len(act), chunk):
            ids = act[s:s + chunk]; bg, bj, bv, ns = one_step(margin, x[ids], X0[ids], g[ids], feats, TH, B, K); scored += ns
            imp = bj >= 0; ii = ids[imp]; x[ii, bj[imp]] = bv[imp]; g[ii] = bg[imp]; stalled[ids[~imp]] = True; active[ids[~imp]] = False
        active &= (g <= 0); log.append((it + 1, len(act), scored, time.time() - t0))
        if verbose: print(f'      round {it + 1:>2}: active rows {len(act):>6} | candidates scored {scored:>10,} | {time.time() - t0:5.1f}s | evaded so far {(g > 0).mean():.3f}', flush=True)
    return x, g, init_evaded, stalled, log
print('white-box attack defined')

## 3. Metrics (as NB4e) and the NB4 oracle protocol

In [ ]:
LAB = list(range(6))
def full_metrics(pred_atk, benign_pred):
    pred = np.empty(len(y_test), dtype=np.int64); pred[y_test == 0] = benign_pred; pred[atk_te] = pred_atk
    return {'weighted_f1': float(f1_score(y_test, pred, average='weighted')), 'macro_f1': float(f1_score(y_test, pred, average='macro')),
            'per_class_f1': f1_score(y_test, pred, average=None, labels=LAB), 'per_class_recall': recall_score(y_test, pred, average=None, labels=LAB),
            'benign_precision': float(precision_score(y_test, pred, labels=[0], average=None)[0]), 'attack_detected': float((pred_atk != 0).mean())}
def per_class_rate(b): return np.array([b[y_atk_test == c].mean() for c in range(1, 6)])
def summarise(x_adv, X0, cpu_clf, benign_pred, init_evaded, stalled, log, g_final, gpu_pred):
    pred = cpu_clf.predict(x_adv); ev = pred == 0; d = np.abs(x_adv - X0); changed = (d > 0).sum(1)
    return {'evaded': ev.copy(), 'per_class_evasion': per_class_rate(ev), 'overall_evasion': float(ev.mean()), 'clean_benign_rate': float(init_evaded.mean()), 'per_class_clean_benign': per_class_rate(init_evaded),
            **full_metrics(pred, benign_pred), 'features_changed_mean': float(changed.mean()), 'features_changed_median': float(np.median(changed[ev])) if ev.any() else float('nan'),
            'max_abs_change_mean': float(d.max(1).mean()), 'max_abs_change_max': float(d.max()), 'rows_stalled': float((stalled & ~ev).mean()), 'rows_iter_limit': float((~stalled & ~ev).mean()),
            'rounds': len(log), 'search_time_s': float(sum(l[3] for l in log)), 'candidates_scored': int(sum(l[2] for l in log)),
            'search_vs_cpu_oracle_disagree': float(((g_final > 0) != ev).mean()), 'gpu_cpu_label_disagree': float((gpu_pred != pred).mean())}
print('metrics defined')

## 4. Self-test (first seed only, before the real run)
Two checks that the candidate generation is right: (1) **exhaustive comparison** -- for random rows and features, the best single move found by `one_step` (with every threshold allowed) must equal the best over an independent enumeration of *all* cells (interval ends and midpoints between consecutive thresholds); (2) **constraints** -- after a short attack every coordinate stays inside the L-infinity ball and the training range.

In [ ]:
def selftest(clf, margin, TH, seed=0):
    rng = np.random.RandomState(seed); rows = rng.choice(len(X_atk_test), 60, replace=False); X0 = X_atk_test[rows]; feats = rng.choice(CYBER_IDX, 12, replace=False)
    g0 = gap_of(margin(X0)); worst = 0.0
    for B in [0.1, 0.5, None]:
        bg, bj, bv, _ = one_step(margin, X0.copy(), X0, g0, feats, TH, B, K=10 ** 6)
        for i in range(len(rows)):
            best = g0[i]
            for j in feats:
                T = TH.get(int(j))
                if T is None: continue
                a, b = interval(X0[i:i + 1, j], int(j), B); a, b = float(a[0]), float(b[0]); inner = T[(T > a) & (T < b)]
                pts = np.concatenate([[a, b], (inner[:-1] + inner[1:]) / 2 if len(inner) > 1 else [], inner, np.nextafter(inner, np.float32(-np.inf))]).astype(np.float32)
                pts = pts[(pts >= a) & (pts <= b)]
                Xc = np.repeat(X0[i:i + 1], len(pts), 0); Xc[:, j] = pts; best = max(best, gap_of(margin(Xc)).max())
            worst = max(worst, best - bg[i])
            assert bg[i] >= best - 1e-4, f'one_step missed a better move (row {i}, budget {B}): {bg[i]} < {best}'
    # constraints after a short attack
    sub = X_atk_test[rng.choice(len(X_atk_test), 400, replace=False)]
    for B in [0.1, 0.5]:
        xa, ga, ie, st, lg = whitebox(margin, sub, CYBER_IDX, TH, B, max_iters=4, verbose=False)
        d = np.abs(xa - sub); assert d[:, CYBER_IDX].max() <= B + 1e-6, 'L-inf budget violated'
        other = np.setdiff1d(np.arange(N_FEAT), CYBER_IDX); assert d[:, other].max() == 0, 'a non-controllable feature changed'
        lo_ok = (xa >= np.minimum(FEAT_LO, sub) - 1e-6) & (xa <= np.maximum(FEAT_HI, sub) + 1e-6); assert lo_ok.all(), 'left the training range'
        assert (ga >= gap_of(margin(sub)) - 1e-9).all(), 'margin gap decreased'
    print(f'self-test passed (largest shortfall of one_step vs full enumeration: {max(worst, 0):.2e}); budget, range, non-controllable features and monotone margin all hold')

## 5. Per-seed run (resumable per seed and setting)

In [ ]:
R = {}; INFO = {}; t_all = time.time()
for seed in SEEDS:
    pf = f'attack1w_seed{seed}_{PIPELINE_VERSION}.pkl'
    cfg_fp = repr((best_params, K_CAND, MAX_ITERS, seed, X_atk_test.shape, float(X_atk_test[:5].sum()), 'nb3c-v2'))
    R[seed] = {}; INFO[seed] = {}
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp: R[seed], INFO[seed] = saved['R'], saved['info']
    todo = [s for s in SETTINGS_TO_RUN if s not in R[seed]]
    if not todo: print(f'seed {seed}: all settings found -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}: {len(todo)} setting(s) to run', flush=True)
    clf = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); clf.fit(X_train, y_train, verbose=False)
    tmp = f'_tmp_xgb_seed{seed}.json'; clf.save_model(tmp); cpu = xgb.XGBClassifier(); cpu.load_model(tmp); cpu.set_params(device='cpu', n_jobs=-1); os.remove(tmp)
    clean_pred = cpu.predict(X_test); benign_pred = clean_pred[y_test == 0]; margin = Margin(clf); TH = get_thresholds(clf)
    INFO[seed] = {'clean_weighted_f1': float(f1_score(y_test, clean_pred, average='weighted')), 'n_trees': int(len(clf.get_booster().get_dump())), 'n_split_features': len(TH),
                  'n_thresholds_total': int(sum(len(v) for v in TH.values()))}
    print(f'  clean weighted F1 {INFO[seed]["clean_weighted_f1"]:.4f} | trees {INFO[seed]["n_trees"]} | features used by the trees {INFO[seed]["n_split_features"]} | distinct thresholds {INFO[seed]["n_thresholds_total"]}', flush=True)
    if seed == SEEDS[0] and not any(k in R[seed] for k in SETTINGS_TO_RUN): selftest(clf, margin, TH)
    for sname in todo:
        grp, B, _ = SETTINGS[sname]; feats = CYBER_IDX if grp == 'cyber' else ALL_IDX; ts = time.time(); print(f'  [{sname}] attacking {len(X_atk_test)} rows ...', flush=True)
        xa, ga, ie, st, lg = whitebox(margin, X_atk_test, feats, TH, B)
        gpu_pred = clf.predict(xa); R[seed][sname] = summarise(xa, X_atk_test, cpu, benign_pred, ie, st, lg, ga, gpu_pred)
        r = R[seed][sname]
        print(f'  [{sname}] per-class evasion {np.round(r["per_class_evasion"], 3).tolist()} overall {r["overall_evasion"]:.3f} (already benign before: {r["clean_benign_rate"]:.3f}) | full-test F1 {r["weighted_f1"]:.4f} (clean {INFO[seed]["clean_weighted_f1"]:.4f}) | '
              f'features changed/row {r["features_changed_mean"]:.1f} | stalled {r["rows_stalled"]:.3f} iter-limit {r["rows_iter_limit"]:.3f} | {(time.time() - ts) / 60:.1f} min', flush=True)
        joblib.dump({'fp': cfg_fp, 'R': R[seed], 'info': INFO[seed]}, pf)
    print(f'  seed {seed} done in {(time.time() - t0) / 60:.1f} min', flush=True)
print(f'Total {(time.time() - t_all) / 60:.1f} min')

## 6. Results: white-box evasion and full-test F1 (mean +/- std over seeds)

In [ ]:
atk_names = class_names[1:]
def have(s_): return [s for s in SEEDS if s_ in R[s]]
def arr(s_, f): return np.stack([np.asarray(R[s][s_][f]) for s in have(s_)])
sd = lambda x: x.std(0, ddof=1) if len(x) > 1 else np.zeros(x.shape[1:])
rows = []
for sname in SETTINGS_TO_RUN:
    if not have(sname): continue
    a = arr(sname, 'per_class_evasion'); o = arr(sname, 'overall_evasion'); n = len(a); w = arr(sname, 'weighted_f1')
    row = {'setting': sname, 'seeds': n}; row.update({c: f'{a[:, i].mean():.3f} +/- {sd(a)[i]:.3f}' for i, c in enumerate(atk_names)})
    row.update({'ALL': f'{o.mean():.3f} +/- {(o.std(ddof=1) if n > 1 else 0):.3f}', 'full-test F1': f'{w.mean():.4f} +/- {(w.std(ddof=1) if n > 1 else 0):.4f}', 'benign precision': f'{arr(sname, "benign_precision").mean():.3f}',
                'attack rows labelled non-benign': f'{arr(sname, "attack_detected").mean():.3f}', 'features changed/row': f'{arr(sname, "features_changed_mean").mean():.1f}',
                'stalled': f'{arr(sname, "rows_stalled").mean():.3f}', 'iter-limit': f'{arr(sname, "rows_iter_limit").mean():.3f}'}); rows.append(row)
wb_table = pd.DataFrame(rows); pd.set_option('display.width', 320); print(wb_table.to_string(index=False))
cl = np.stack([np.asarray(R[s][SETTINGS_TO_RUN[0]]['per_class_clean_benign']) for s in SEEDS if SETTINGS_TO_RUN[0] in R[s]])
print('\nShare of each attack class ALREADY classified benign before any perturbation (evasion at zero effort):', dict(zip(atk_names, np.round(cl.mean(0), 3))))
dis = max(max(R[s][k]['gpu_cpu_label_disagree'] for k in R[s]) for s in SEEDS); dis2 = max(max(R[s][k]['search_vs_cpu_oracle_disagree'] for k in R[s]) for s in SEEDS)
print(f'Largest share of rows where the GPU search model and the CPU oracle disagree after the attack: labels {dis:.2e}, evaded-flag {dis2:.2e}')

## 6b. Burst-clustered 95% intervals (new in v6b)
Resample attack bursts (class, PLC, timestamp) with replacement, 2,000 times; for each resample take the share of evaded rows, averaged over the 5 seeds. Same procedure as NB4 and the Morris/TU Delft notebooks. Seed-to-seed spread is model variance; this interval is test-set sampling variance.

In [ ]:
N_BOOT = 2000; rng = np.random.RandomState(42); ub = np.unique(test_burst); bi = np.searchsorted(ub, test_burst); nb_ = len(ub); n_atk = np.bincount(bi, minlength=nb_).astype(float)
idx = rng.randint(0, nb_, size=(N_BOOT, nb_)); BCI = {}
for sname in SETTINGS_TO_RUN:
    if len(have(sname)) != len(SEEDS): continue
    boots = []
    for s in SEEDS:
        ev = np.bincount(bi, weights=R[s][sname]['evaded'].astype(float), minlength=nb_); boots.append(ev[idx].sum(1) / n_atk[idx].sum(1))
    pooled = np.mean(boots, 0); BCI[sname] = (float(arr(sname, 'overall_evasion').mean()), *[float(v) for v in np.percentile(pooled, [2.5, 97.5])])
print(f'{nb_} attack bursts for {len(y_atk_test):,} rows\n')
print(f'{"setting":<12}{"evasion":>9}{"95% lo":>9}{"95% hi":>9}')
for k, (m, lo, hi) in BCI.items(): print(f'{k:<12}{m:>9.4f}{lo:>9.4f}{hi:>9.4f}')


## 7. White-box versus PPO, random search and PGD (paired over seeds)
PPO and random search come from `attack2e_results_v6.pkl` (NB4e): evasion = 1 - share of attack rows still labelled non-benign. For `cyber_inf` the matching PPO reference is the range-clipped unconstrained agent. PGD comes from NB3b's saved confusion matrices (all 93 features, 0.1 sigma, transferred from a surrogate; it also perturbs benign rows, so only its evasion on attack rows is comparable, not its F1).

In [ ]:
ref = joblib.load(f'attack2e_results_{PIPELINE_VERSION}.pkl')['per_seed'] if os.path.exists(f'attack2e_results_{PIPELINE_VERSION}.pkl') else None
def ref_vals(method, tag):
    if ref is None or not all(s in ref and f'{method}|{tag}' in ref[s] for s in SEEDS): return None
    return {'evasion': np.array([1 - ref[s][f'{method}|{tag}']['attack_detected'] for s in SEEDS]), 'f1': np.array([ref[s][f'{method}|{tag}']['weighted_f1'] for s in SEEDS])}
def ptest(A, B_): d = A - B_; return 'n/a' if len(d) < 2 or d.std(ddof=1) == 0 else f'{stats.ttest_rel(A, B_).pvalue:.4f}'
cmp_rows = []
for sname in SETTINGS_TO_RUN:
    if len(have(sname)) != len(SEEDS) or SETTINGS[sname][2] is None: continue
    tag = SETTINGS[sname][2]; we, wf = arr(sname, 'overall_evasion'), arr(sname, 'weighted_f1')
    for lab, meth in [('PPO (hard-label)', 'PPO_clip' if tag == 'unconstrained' else 'PPO'), ('random search', 'Random')]:
        rv = ref_vals(meth, tag)
        if rv is None: continue
        cmp_rows.append({'setting': sname, 'vs': lab, 'white-box evasion': f'{we.mean():.3f}', 'other evasion': f'{rv["evasion"].mean():.3f}', 'diff': f'{(we - rv["evasion"]).mean():+.3f}', 'p (paired t)': ptest(we, rv['evasion']),
                         'white-box F1': f'{wf.mean():.4f}', 'other F1': f'{rv["f1"].mean():.4f}'})
# PGD (NB3b), all-feature 0.1 sigma
pgd = None
if os.path.exists(f'attack1_confusion_{PIPELINE_VERSION}.pkl'):
    cm = joblib.load(f'attack1_confusion_{PIPELINE_VERSION}.pkl')['counts_pgd']
    pgd = {'evasion': np.array([cm[s][1:, 0].sum() / cm[s][1:].sum() for s in SEEDS]), 'per_class': np.stack([cm[s][1:, 0] / cm[s][1:].sum(1) for s in SEEDS])}
    if len(have('all93_0.1')) == len(SEEDS):
        we = arr('all93_0.1', 'overall_evasion'); cmp_rows.append({'setting': 'all93_0.1', 'vs': 'PGD on surrogate (NB3b)', 'white-box evasion': f'{we.mean():.3f}', 'other evasion': f'{pgd["evasion"].mean():.3f}', 'diff': f'{(we - pgd["evasion"]).mean():+.3f}',
                         'p (paired t)': ptest(we, pgd['evasion']), 'white-box F1': '-', 'other F1': '-'})
cmp_table = pd.DataFrame(cmp_rows); print(cmp_table.to_string(index=False) if len(cmp_table) else 'reference files missing -- comparison skipped')
if pgd is not None and len(have('all93_0.1')) == len(SEEDS):
    a = arr('all93_0.1', 'per_class_evasion'); print('\nPer-class evasion, all-feature 0.1 sigma:  white-box', dict(zip(atk_names, np.round(a.mean(0), 3))), '| PGD (surrogate transfer)', dict(zip(atk_names, np.round(pgd['per_class'].mean(0), 3))))
inf = have('cyber_inf')
if inf: print(f'\nSanity (unlimited budget, training range only): overall evasion {arr("cyber_inf", "overall_evasion").mean():.4f}, worst class {arr("cyber_inf", "per_class_evasion").mean(0).min():.4f}  (a complete white-box search would be ~1.0)')

## 8. Figure and save

In [ ]:
tags = ['cyber_0.1', 'cyber_0.2', 'cyber_0.3', 'cyber_0.5', 'cyber_1.0']; xl = ['0.1', '0.2', '0.3', '0.5', '1.0']; xs = np.arange(len(tags))
fig, ax = plt.subplots(1, 3, figsize=(19, 4.8))
def line(ax_, key_fn, label, fmt):
    ys = [key_fn(t) for t in tags]
    if all(y is not None for y in ys): ax_.plot(xs, [y[0] for y in ys], fmt, label=label); ax_.fill_between(xs, [y[0] - y[1] for y in ys], [y[0] + y[1] for y in ys], alpha=0.15)
wb = lambda f: (lambda t: (arr(t, f).mean(), arr(t, f).std(ddof=1)) if len(have(t)) == len(SEEDS) else None)
line(ax[0], lambda t: wb('overall_evasion')(t), 'white-box (trees)', 'o-')
for meth, fmt in [('PPO', 's--'), ('Random', '^:')]:
    ys = [ref_vals(meth, SETTINGS[t][2]) for t in tags]
    if all(y is not None for y in ys): ax[0].plot(xs, [y['evasion'].mean() for y in ys], fmt, label=meth)
ax[0].set_xticks(xs); ax[0].set_xticklabels(xl); ax[0].set_xlabel('cumulative cyber budget (sigma)'); ax[0].set_ylabel('overall evasion'); ax[0].set_title('Evasion: white-box vs PPO vs random'); ax[0].legend(); ax[0].grid(alpha=.3)
line(ax[1], lambda t: wb('weighted_f1')(t), 'white-box (trees)', 'o-')
for meth, fmt in [('PPO', 's--'), ('Random', '^:')]:
    ys = [ref_vals(meth, SETTINGS[t][2]) for t in tags]
    if all(y is not None for y in ys): ax[1].plot(xs, [y['f1'].mean() for y in ys], fmt, label=meth)
ax[1].axhline(np.mean([INFO[s]['clean_weighted_f1'] for s in SEEDS if 'clean_weighted_f1' in INFO[s]]), color='k', ls=':', label='clean')
ax[1].set_xticks(xs); ax[1].set_xticklabels(xl); ax[1].set_xlabel('cumulative cyber budget (sigma)'); ax[1].set_ylabel('full-test weighted F1'); ax[1].set_title('Detector F1 under attack'); ax[1].legend(); ax[1].grid(alpha=.3)
for k, t in enumerate(tags):
    if len(have(t)) == len(SEEDS): a = arr(t, 'per_class_evasion'); ax[2].bar(np.arange(5) + (k - 2) * 0.16, a.mean(0), 0.16, label=xl[k] + ' sigma')
ax[2].set_xticks(np.arange(5)); ax[2].set_xticklabels(atk_names, rotation=15); ax[2].set_ylabel('white-box evasion'); ax[2].set_title('Per-class white-box evasion'); ax[2].legend(fontsize=8); ax[2].grid(alpha=.3, axis='y')
plt.tight_layout(); plt.savefig(f'attack1w_whitebox_{PIPELINE_VERSION}.png', dpi=150); plt.show()

joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'SETTINGS': SETTINGS, 'K_CAND': K_CAND, 'MAX_ITERS': MAX_ITERS, 'class_names': class_names, 'wb_table': wb_table, 'burst_ci': BCI, 'n_bursts': nb_, 'cmp_table': cmp_table, 'info': INFO,
             'per_seed': {s: dict(R[s]) for s in SEEDS}}, f'attack1w_results_{PIPELINE_VERSION}.pkl')
wb_table.to_csv(f'attack1w_results_table_{PIPELINE_VERSION}.csv', index=False); cmp_table.to_csv(f'attack1w_comparison_{PIPELINE_VERSION}.csv', index=False)
json.dump({'notebook': 'NB3c', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'K_CAND': K_CAND, 'MAX_ITERS': MAX_ITERS, 'xgboost': xgb.__version__, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb3c_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'attack1w_results_{PIPELINE_VERSION}.pkl', f'attack1w_results_table_{PIPELINE_VERSION}.csv', f'attack1w_comparison_{PIPELINE_VERSION}.csv', f'attack1w_whitebox_{PIPELINE_VERSION}.png', f'nb3c_run_manifest_{PIPELINE_VERSION}.json']:
    print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)